In [1]:
import json
from pathlib import Path
import re

# huggingface dataset class
from datasets import Dataset

# rapidfire ai wrapper
from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

In [2]:
def load_schema(db_id, schemas_dir='./schemas'):
    # change to the correct file name
    fname = db_id.replace(' ', '_').replace('/', '_') + '.json'
    # load the json file
    with open(f'{schemas_dir}/{fname}') as f:
        s = json.load(f)
    # to store table name and column mapping
    schema = {t: [] for t in s['table_names_original']}
    for tidx, cname in s['column_names_original']:
        # skip the synthetic '*' entry
        if tidx == -1:
            continue
        schema[s['table_names_original'][tidx]].append(cname)
    # {table_name: [col, col, ...]}
    return schema

In [3]:
def build_prompt(question, schema_text):
    return (
        f'Question:\n{question}\n'
        f'Schema:\n{schema_text}\n'
        'Return only schema_links JSON.'
    )

In [4]:
def formatting_function(row):
    SYSTEM_PROMPT = '''You are a schema-linking model.\n
    Given a database schema and a natural-language question, output only valid JSON.
    The JSON must map table names to lists of referenced column names.
    Include tables even when no specific columns are referenced, using an empty list.
    Do not include explanations, markdown, or extra text.

    Example output:
    {"TableName1": ["Col1", "Col2"], "TableName2": ["Col3"]}
    '''
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['prompt_text']}
        ],
        'completion': [
            {'role': 'assistant', 'content': row['completion_text']}
        ]
    }

In [5]:
def build_dataset(dataset_file, schemas_dir='./schemas'):
    # load the examples
    with open(dataset_file, 'r') as f:
        examples = json.load(f)
    # to store a list of formatted example for the dataset
    formatted = []
    for example in examples:
        # load the schema that corresponds to the db_id of the example
        schema = load_schema(db_id=example['db_id'], schemas_dir=schemas_dir)
        schema_text = '\n'.join(
            f'- {table}: {", ".join(columns)}'
            for table, columns in schema.items()
        )
        # build the prompt text by combining schema and question
        prompt_text = build_prompt(question=example['question'], schema_text=schema_text)
        # convert the target schema links into a JSON string
        completion_text = json.dumps(example['schema_links'], sort_keys=True)
        # add them to the dict
        example['prompt_text'] = prompt_text
        example['completion_text'] = completion_text
        # add the formatted example to the list
        formatted.append(example)
    return formatted

In [6]:
def extract_schema_links(text):
    import json, re

    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Prefer the first JSON-looking object in the generated output.
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        return {}

    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}

    if not isinstance(obj, dict):
        return {}

    cleaned = {}
    for table, cols in obj.items():
        if isinstance(cols, list):
            cleaned[str(table)] = [str(c) for c in cols]
        else:
            cleaned[str(table)] = []

    return cleaned

In [7]:
def compute_metrics(eval_preds):
    import re, json
    # separte predictions and references
    predictions, references = eval_preds
    # try to extract json
    predictions = extract_schema_links(predictions)
    exact_matches = 0
    counts = {
        'table': {'tp': 0, 'fp': 0, 'fn': 0},
        'column': {'tp': 0, 'fp': 0, 'fn': 0},
    }
    # for each example
    for pred, ref in zip(predictions, references):
        # parse the prediction
        try:
            pred_json = json.loads(pred)
        except:
            # try find the json formatting
            match = re.search(r'\{.*\}', str(pred), re.S)
            pred_json = json.loads(match.group(0)) if match else {}
        # parse the reference
        try:
            ref_json = json.loads(ref)
        except:
            ref_json = {}
        # make sure both parsed objects are dictionaries
        if not isinstance(pred_json, dict):
            pred_json = {}
        if not isinstance(ref_json, dict):
            ref_json = {}
        # if the prediction and reference match exactly
        if pred_json == ref_json:
            exact_matches += 1
        # predicted and reference table names
        pred_tables = set(pred_json.keys())
        ref_tables = set(ref_json.keys())
        # to store all the key column pairs
        pred_columns, ref_columns = set(), set()
        for table, columns in pred_json.items():
            if isinstance(columns, list):
                for column in columns:
                    pred_columns.add(f'{table}.{column}')
        for table, columns in ref_json.items():
            if isinstance(columns, list):
                for column in columns:
                    ref_columns.add(f'{table}.{column}')
        # update table-level tp, fp, fn
        counts['table']['tp'] += len(pred_tables & ref_tables)
        counts['table']['fp'] += len(pred_tables - ref_tables)
        counts['table']['fn'] += len(ref_tables - pred_tables)
        # update column-level tp, fp, fn
        counts['column']['tp'] += len(pred_columns & ref_columns)
        counts['column']['fp'] += len(pred_columns - ref_columns)
        counts['column']['fn'] += len(ref_columns - pred_columns)
    
    # calculate table precision, recall, f1 from total table counts
    table_tp = counts['table']['tp']
    table_fp = counts['table']['fp']
    table_fn = counts['table']['fn']
    table_precision = table_tp / (table_tp + table_fp) if table_tp + table_fp > 0 else 0
    table_recall = table_tp / (table_tp + table_fn) if table_tp + table_fn > 0 else 0
    table_f1 = (
        2 * table_precision * table_recall / (table_precision + table_recall)
        if table_precision + table_recall > 0
        else 0
    )

    # calculate column precision, recall, f1 from total column counts
    column_tp = counts['column']['tp']
    column_fp = counts['column']['fp']
    column_fn = counts['column']['fn']
    column_precision = column_tp / (column_tp + column_fp) if column_tp + column_fp > 0 else 0
    column_recall = column_tp / (column_tp + column_fn) if column_tp + column_fn > 0 else 0
    column_f1 = (
        2 * column_precision * column_recall / (column_precision + column_recall)
        if column_precision + column_recall > 0
        else 0
    )
    return {
        'exact_match': round(exact_matches / len(references), 4),

        'table_precision': round(table_precision, 4),
        'table_recall': round(table_recall, 4),
        'table_f1': round(table_f1, 4),

        'column_precision': round(column_precision, 4),
        'column_recall': round(column_recall, 4),
        'column_f1': round(column_f1, 4),

        'num_predictions': len(predictions),
        'num_references': len(references),
    }

In [8]:
# LoRA PEFT configs lite with different adapter capacities
peft_configs_lite = List([
    # RFLoraConfig(
    #     # lora rank
    #     r=8,
    #     # normally 2 x rank
    #     lora_alpha=16,
    #     lora_dropout=0.05,
    #     # target on query and value projections
    #     target_modules=['q_proj', 'v_proj'],
    #     bias='none'
    # ),
    RFLoraConfig(
        # lora rank
        r=32,
        # normally 2 x rank
        lora_alpha=64,
        lora_dropout=0.05,
        # target on query, key, value projections and output projection
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        bias='none'
    )
])

config_set_lite = List([
    RFModelConfig(
        model_name='google/gemma-2-2b',
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=2e-5,
            lr_scheduler_type='cosine',
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            max_steps=128,
            warmup_ratio=0.05,
            max_grad_norm=.2,
            gradient_accumulation_steps=4,
            logging_steps=4,
            eval_strategy='steps',
            eval_steps=4,
            fp16=True,
            save_strategy="steps",
            save_steps=16,
            save_total_limit=3,
            output_dir='./checkpoint'
        ),
        model_type='causal_lm',
        model_kwargs={'device_map': 'auto', 'torch_dtype': 'auto', 'use_cache': False},
        formatting_func=formatting_function,
        compute_metrics=compute_metrics,
        generation_config={
            'max_new_tokens': 256,
            'temperature': 0.8,
            'top_p': 0.9,
            'top_k': 30,
            'repetition_penalty': 1.05,
        }
    ),
    RFModelConfig(
        model_name='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        peft_config=peft_configs_lite,
        training_args=RFSFTConfig(
            learning_rate=2e-5,
            lr_scheduler_type='cosine',
            per_device_train_batch_size=4,
            per_device_eval_batch_size=4,
            max_steps=128,
            warmup_ratio=0.05,
            max_grad_norm=.2,
            gradient_accumulation_steps=4,
            logging_steps=4,
            eval_strategy='steps',
            eval_steps=4,
            fp16=True,
        ),
        model_type='causal_lm',
        model_kwargs={'device_map': 'auto', 'torch_dtype': 'auto', 'use_cache': False},
        formatting_func=formatting_function,
        compute_metrics=compute_metrics,
        generation_config={
            'max_new_tokens': 256,
            'temperature': 0.8,
            'top_p': 0.9,
            'top_k': 30,
            'repetition_penalty': 1.05,
        }
    )
])

In [9]:
def create_model(model_config): 
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForMaskedLM
    # extract model configurations to initialize and load a pretrained model
    model_name = model_config['model_name']
    model_type = model_config['model_type']
    model_kwargs = model_config['model_kwargs']
    # load the pretrained model
    if model_type == 'causal_lm':
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == 'seq2seq_lm':
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, **model_kwargs)
    elif model_type == 'masked_lm':
        model = AutoModelForMaskedLM.from_pretrained(model_name, **model_kwargs)
    else:
        return None
    # load the pretrained tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return (model,tokenizer)

In [10]:
config_group = RFGridSearch(
    configs=config_set_lite,
    trainer_type='SFT'
)

In [11]:
# build training and valiation datasets
train_examples = build_dataset(dataset_file='./train.json', schemas_dir='./schemas')
val_examples = build_dataset(dataset_file='./validation.json', schemas_dir='./schemas')
# convert lists into HuggingFace Dataset objects
train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

In [12]:
print(formatting_function(train_dataset[0])['prompt'])
print(formatting_function(train_dataset[0])['completion'])

[{'role': 'system', 'content': 'You are a schema-linking model.\n\n    Given a database schema and a natural-language question, output only valid JSON.\n    The JSON must map table names to lists of referenced column names.\n    Include tables even when no specific columns are referenced, using an empty list.\n    Do not include explanations, markdown, or extra text.\n\n    Example output:\n    {"TableName1": ["Col1", "Col2"], "TableName2": ["Col3"]}\n    '}, {'role': 'user', 'content': 'Question:\nShow a count of injuries by body region where the injury severity is critical. The lookup code for critical injury is 5.\nSchema:\n- AIRBAG: CASEID, PSU, CASENO, CASENUMBER, CATEGORY, VEHNO, BAGNO, BAGLOCATION, BAGSTATUS, BAGTYPE, BAGDEPLOY, BAGMALFUNCTION, BAGFLAPSOPEN, BAGFLAPSDAM, BAGDAMAGE, BAGDAMSOURCE, PREVCRASH, PRIORMAINT, DEPLOYEVENT, BAGDVTOTAL, BAGDVLONG, BAGDVLAT, BAGDVENERGY, BAGDVSPEED, BAGDVBES, BAGDVEST, BAGDVRANK, CDCFORDEPLOY, CASEWGT, PSUSTRAT, VERSION\n- CHILDSEAT: CASEID

In [13]:
# initialize a training experiment
experiment = Experiment(experiment_name='schema-linking-qwen-baseline', mode='fit')

An experiment with the same name already exists. Created a new experiment 'schema-linking-qwen-baseline_4' with Experiment ID: 5 and Metric Experiment ID: 5 at /home/chc016/rapidfireai/rapidfire_experiments/schema-linking-qwen-baseline_4


In [14]:
# run multi config training
experiment.run_fit(config_group, create_model_fn=create_model, train_dataset=train_dataset, eval_dataset=val_dataset, num_chunks=4, seed=42)

INFO 05-30 15:36:12 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-30 15:36:25 [__init__.py:216] Automatically detected platform cuda.
Run 1 has failed: Cannot use chat template functions because tokenizer.chat_template is not set and no template argument was passed! For information about writing templates and setting the tokenizer.chat_template attribute, please see the documentation at https://huggingface.co/docs/transformers/main/en/chat_templatingTraceback (most recent call last):
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/rapidfireai/fit/backend/worker.py", line 676, in serve_forever
    self.run_fit(
  File "/opt/conda/envs/cse234/lib/python3.12/site-packages/rapidfireai/fit/backend/worker.py", line 354, in run_fit
    trainer_instance, base_model_name = create_trainer_instance(
                                        ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/cse234/lib/python3.12/sit

In [15]:
# end the experiment
experiment.end()

Experiment schema-linking-qwen-baseline_4 ended
Workers stopped
